In [1]:
import pandas as pd
import numpy as np
import re

from sklearn.model_selection import TimeSeriesSplit
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor, StackingRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error
from sklearn.base import clone

import statsmodels.api as sm
import warnings
warnings.filterwarnings("ignore")


In [2]:
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', None)

In [6]:
df = pd.read_csv("./아마도최종데이터.csv")
df.head()

,Date,brent_close,wti_close,brent_wti_spread,brent_ret_1d,brent_ret_5d,brent_ret_20d,brent_ma_5,brent_ma_20,brent_ma_60,brent_vol_5d,high_low_range,crude_stock_level,gas_stock_level,dist_stock_level,crude_stock_wow_change,gas_stock_wow_change,dist_stock_wow_change,crude_stock_vs_5yr_avg,gas_stock_vs_5yr_avg,dist_stock_vs_5yr_avg,prod_weekly,prod_4w_ma,prod_wow_change,crude_exports,crude_imports,import_4w_ma,net_imports,refinery_run_rate,wti_mm_net_long,wti_mm_net_long_ratio,wti_producer_hedge_ratio,wti_mm_position_change_wow,wti_sentiment_position,DXY_DX-Y.NYB,XLE_XLE,VDE_VDE,IXC_IXC,"('DXY', 'DX-Y.NYB')_ret_1d","('XLE', 'XLE')_ret_1d","('VDE', 'VDE')_ret_1d","('IXC', 'IXC')_ret_1d",us10y_yield,us2y_yield,term_spread,bdi_price,bdi_open,bdi_high,bdi_low,bdi_change_%,wti_mm_net_long_is_update,wti_mm_net_long_shock_raw,wti_mm_net_long_days_since_update,wti_mm_net_long_avail,wti_mm_net_long_shock_decay,wti_mm_net_long_ratio_is_update,wti_mm_net_long_ratio_shock_raw,wti_mm_net_long_ratio_days_since_update,wti_mm_net_long_ratio_avail,wti_mm_net_long_ratio_shock_decay,wti_producer_hedge_ratio_is_update,wti_producer_hedge_ratio_shock_raw,wti_producer_hedge_ratio_days_since_update,wti_producer_hedge_ratio_avail,wti_producer_hedge_ratio_shock_decay,wti_mm_position_change_wow_is_update,wti_mm_position_change_wow_shock_raw,wti_mm_position_change_wow_days_since_update,wti_mm_position_change_wow_avail,wti_mm_position_change_wow_shock_decay,wti_sentiment_position_is_update,wti_sentiment_position_shock_raw,wti_sentiment_position_days_since_update,wti_sentiment_position_avail,wti_sentiment_position_shock_decay,event_large_move,flag_crude_stock_wow_change,flag_prod_wow_change,flag_wti_mm_position_change_wow,wti_producer_hedge_ratio_lag3,wti_producer_hedge_ratio_lag5,spread_regime,spread_regime_code,spread_extreme,spread_chg_5d,crude_vs5_regime,crude_vs5_regime_code,crude_vs5_extreme,crude_vs5_zscore,news_total_count,clust_0_count,clust_1_count,clust_2_count,clust_3_count,clust_4_count,clust_5_count,clust_6_count,clust_7_count,clust_8_count,clust_9_count,clust_10_count,clust_11_count,clust_12_count,clust_13_count,clust_14_count,clust_15_count,clust_16_count,clust_17_count,clust_18_count,clust_19_count,clust_20_count,clust_21_count,clust_22_count,clust_23_count,clust_24_count,clust_25_count,clust_26_count,clust_27_count,clust_28_count,clust_29_count,clust_0_impact_mean,clust_1_impact_mean,clust_2_impact_mean,clust_3_impact_mean,clust_4_impact_mean,clust_5_impact_mean,clust_6_impact_mean,clust_7_impact_mean,clust_8_impact_mean,clust_9_impact_mean,clust_10_impact_mean,clust_11_impact_mean,clust_12_impact_mean,clust_13_impact_mean,clust_14_impact_mean,clust_15_impact_mean,clust_16_impact_mean,clust_17_impact_mean,clust_18_impact_mean,clust_19_impact_mean,clust_20_impact_mean,clust_21_impact_mean,clust_22_impact_mean,clust_23_impact_mean,clust_24_impact_mean,clust_25_impact_mean,clust_26_impact_mean,clust_27_impact_mean,clust_28_impact_mean,clust_29_impact_mean,clust_0_impact_sum,clust_1_impact_sum,clust_2_impact_sum,clust_3_impact_sum,clust_4_impact_sum,clust_5_impact_sum,clust_6_impact_sum,clust_7_impact_sum,clust_8_impact_sum,clust_9_impact_sum,clust_10_impact_sum,clust_11_impact_sum,clust_12_impact_sum,clust_13_impact_sum,clust_14_impact_sum,clust_15_impact_sum,clust_16_impact_sum,clust_17_impact_sum,clust_18_impact_sum,clust_19_impact_sum,clust_20_impact_sum,clust_21_impact_sum,clust_22_impact_sum,clust_23_impact_sum,clust_24_impact_sum,clust_25_impact_sum,clust_26_impact_sum,clust_27_impact_sum,clust_28_impact_sum,clust_29_impact_sum,news_emb_0,news_emb_1,news_emb_2,news_emb_3,news_emb_4,news_emb_5,news_emb_6,news_emb_7,news_emb_8,news_emb_9,news_emb_10,news_emb_11,news_emb_12,news_emb_13,news_emb_14,news_emb_15,news_emb_16,news_emb_17,news_emb_18,news_emb_19,news_emb_20,news_emb_21,news_emb_22,news_emb_23,news_emb_24,news_emb_25,news_emb_26,news_emb_27,news_emb_28,news_emb_29,news_emb_30,news_emb_31,news_emb_32,news_emb_33,news_emb_34,news_emb

In [7]:
import re
import numpy as np

# 1) cluster prefix 정의
count_cols = [c for c in df.columns if re.match(r"clust_\d+_count", c)]
mean_cols  = [c for c in df.columns if re.match(r"clust_\d+_impact_mean", c)]
sum_cols   = [c for c in df.columns if re.match(r"clust_\d+_impact_sum", c)]

# 클러스터 번호 추출
cluster_ids = sorted([int(re.findall(r'\d+', c)[0]) for c in count_cols])

# 2) row별 dominant cluster 선택
df["clust_id"] = df[count_cols].idxmax(axis=1).str.extract(r"clust_(\d+)_count").astype(int)

# 3) 선택된 cluster의 값만 가져오기
def pick_value(row, prefix_list, col_type):
    cid = row["clust_id"]
    col = f"clust_{cid}_{col_type}"
    return row[col]

df["clust_count"] = df.apply(lambda r: pick_value(r, count_cols, "count"), axis=1)
df["clust_mean"]  = df.apply(lambda r: pick_value(r, mean_cols,  "impact_mean"), axis=1)
df["clust_sum"]   = df.apply(lambda r: pick_value(r, sum_cols,   "impact_sum"), axis=1)

# 4) 필요하다면 기존 컬럼 삭제
df = df.drop(columns = count_cols + mean_cols + sum_cols)
df.head()

,Date,brent_close,wti_close,brent_wti_spread,brent_ret_1d,brent_ret_5d,brent_ret_20d,brent_ma_5,brent_ma_20,brent_ma_60,brent_vol_5d,high_low_range,crude_stock_level,gas_stock_level,dist_stock_level,crude_stock_wow_change,gas_stock_wow_change,dist_stock_wow_change,crude_stock_vs_5yr_avg,gas_stock_vs_5yr_avg,dist_stock_vs_5yr_avg,prod_weekly,prod_4w_ma,prod_wow_change,crude_exports,crude_imports,import_4w_ma,net_imports,refinery_run_rate,wti_mm_net_long,wti_mm_net_long_ratio,wti_producer_hedge_ratio,wti_mm_position_change_wow,wti_sentiment_position,DXY_DX-Y.NYB,XLE_XLE,VDE_VDE,IXC_IXC,"('DXY', 'DX-Y.NYB')_ret_1d","('XLE', 'XLE')_ret_1d","('VDE', 'VDE')_ret_1d","('IXC', 'IXC')_ret_1d",us10y_yield,us2y_yield,term_spread,bdi_price,bdi_open,bdi_high,bdi_low,bdi_change_%,wti_mm_net_long_is_update,wti_mm_net_long_shock_raw,wti_mm_net_long_days_since_update,wti_mm_net_long_avail,wti_mm_net_long_shock_decay,wti_mm_net_long_ratio_is_update,wti_mm_net_long_ratio_shock_raw,wti_mm_net_long_ratio_days_since_update,wti_mm_net_long_ratio_avail,wti_mm_net_long_ratio_shock_decay,wti_producer_hedge_ratio_is_update,wti_producer_hedge_ratio_shock_raw,wti_producer_hedge_ratio_days_since_update,wti_producer_hedge_ratio_avail,wti_producer_hedge_ratio_shock_decay,wti_mm_position_change_wow_is_update,wti_mm_position_change_wow_shock_raw,wti_mm_position_change_wow_days_since_update,wti_mm_position_change_wow_avail,wti_mm_position_change_wow_shock_decay,wti_sentiment_position_is_update,wti_sentiment_position_shock_raw,wti_sentiment_position_days_since_update,wti_sentiment_position_avail,wti_sentiment_position_shock_decay,event_large_move,flag_crude_stock_wow_change,flag_prod_wow_change,flag_wti_mm_position_change_wow,wti_producer_hedge_ratio_lag3,wti_producer_hedge_ratio_lag5,spread_regime,spread_regime_code,spread_extreme,spread_chg_5d,crude_vs5_regime,crude_vs5_regime_code,crude_vs5_extreme,crude_vs5_zscore,news_total_count,news_emb_0,news_emb_1,news_emb_2,news_emb_3,news_emb_4,news_emb_5,news_emb_6,news_emb_7,news_emb_8,news_emb_9,news_emb_10,news_emb_11,news_emb_12,news_emb_13,news_emb_14,news_emb_15,news_emb_16,news_emb_17,news_emb_18,news_emb_19,news_emb_20,news_emb_21,news_emb_22,news_emb_23,news_emb_24,news_emb_25,news_emb_26,news_emb_27,news_emb_28,news_emb_29,news_emb_30,news_emb_31,news_emb_32,news_emb_33,news_emb_34,news_emb_35,news_emb_36,news_emb_37,news_emb_38,news_emb_39,news_emb_40,news_emb_41,news_emb_42,news_emb_43,news_emb_44,news_emb_45,news_emb_46,news_emb_47,news_emb_48,news_emb_49,news_emb_50,news_emb_51,news_emb_52,news_emb_53,news_emb_54,news_emb_55,news_emb_56,news_emb_57,news_emb_58,news_emb_59,news_emb_60,news_emb_61,news_emb_62,news_emb_63,news_uuids,clust_id,clust_count,clust_mean,clust_sum
0,2014-01-02,107.779999,95.440002,12.339996,-0.027256,-0.036819,-0.032930,110.790001,110.6705,109.391334,0.011831,0.033680,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.000000,0.000000,0.0,0.00,0.000000,0.0,0.0,0.00,0.0,0.0,0.0,0.00000,0.000000,0.0,0.0,80.629997,55.728363,85.919952,27.177942,0.000000,0.000000,0.000000,0.000000,3.00,0.39,2.61,2113.0,2113.0,2113.0,2113.0,-0.0720,0,0.0,0.0,0,0.0,0,0.0,0.0,0,0.0,0,0.0,0.0,0,0.0,0,0.0,0.0,0,0.0,0,0.0,0.0,0,0.0,0,0,0,0,0.0,0.0,High,1,1,0.0,NaN,0.0,1,0.000000,1.0,-16.985781,1.941138,0.409525,0.716653,1.404438,1.505001,-1.049370,-0.750015,-0.717748,-0.684713,-0.019861,-0.397699,0.108216,1.037515,-0.018697,-0.290782,-0.011142,0.180858,-0.058451,-0.008488,0.300006,0.490811,0.358170,-0.417532,0.461595,-0.015604,-0.014203,0.298583,-0.221794,-0.059309,0.902873,0.280538,0.338775,0.672535,-0.232616,-0.242950,0.079401,-0.132877,0.417334,0.197269,0.065511,0.002837,-0.044323,0.178835,0.334882,-0.299075,0.006763,0.009454,0.458828,-0.403282,0.461847,0.208957,0.177038,-0.108208,-0.052849,-0.502219,-0.685612,-0.246974,-0.458016,-0.169944,0.210896,-0.423135,0.256165,-0.326144,['0d8e95ab-a21d-4755-8ce1-c7ae0190b058'],9,1.0,-0.8,-0.8
1,2014-01-03,106.889999,93.959999,12.930000,-0.008258,-0.045455,-0.050879,109.772000,110.3

In [8]:
mapping = {
    "Low": 0,
    "Mid": 1,
    "High": 2
}

df["spread_regime"] = df["spread_regime"].map(mapping)
mapping = {
    "Tight(낮음)": 0,   # 가장 타이트 → 상승 압력 큼
    "Normal": 1,
    "Loose(높음)": 2    # 가장 느슨함 → 하락 압력 큼
}

df["crude_vs5_regime"] = (
    df["crude_vs5_regime"]
        .map(mapping)
        .astype("float")
)


In [9]:
# Date 정렬
df = df.sort_values("Date").reset_index(drop=True)
df["Date"] = pd.to_datetime(df["Date"])

# --------------------------
# 1) 기본 타깃 정의 (1일 수익률)
# --------------------------
if "brent_ret_1d" in df.columns:
    df["y_ret"] = df["brent_ret_1d"].shift(-1)  # 내일 수익률 예측
else:
    df["y_ret"] = np.log(df["brent_close"]).diff().shift(-1)

# 마지막 행은 shift(-1) 때문에 NaN → 제거 예정
# --------------------------
# 2) 피처 정의
# --------------------------

# (1) 숫자형 피처들
num_cols_base = [
    # 가격/기술
    "brent_close",
    "brent_ret_1d", "brent_ret_5d", "brent_ret_20d",
    "brent_ma_5", "brent_ma_20", "brent_ma_60",
    "brent_vol_5d", "high_low_range",
    
    # 스프레드/재고/생산/수출입
    "brent_wti_spread", "spread_chg_5d",
    "crude_stock_vs_5yr_avg", "crude_vs5_zscore",
    "prod_weekly", "prod_4w_ma", "prod_wow_change",
    "crude_exports", "crude_imports", "import_4w_ma", "net_imports",
    "refinery_run_rate",
    
    # 거시/금리/달러/ETF
    "DXY_DX-Y.NYB",
    "('DXY', 'DX-Y.NYB')_ret_1d",
    "us2y_yield", "us10y_yield", "term_spread",
    "XLE_XLE", "('XLE', 'XLE')_ret_1d",
    "VDE_VDE", "('VDE', 'VDE')_ret_1d",
    "IXC_IXC", "('IXC', 'IXC')_ret_1d",
    
    # BDI
    "bdi_price", "bdi_change_%",
    
    # CFTC shock 기반 심리
    "wti_mm_net_long_ratio_shock_decay",
    "wti_producer_hedge_ratio_shock_decay",
    "wti_mm_position_change_wow_shock_decay",
    
    # 뉴스 전체 개수
    "news_total_count",
]

# (2) 뉴스 클러스터 피처: count + impact_sum만 사용 (mean은 생략)
clust_num_cols = [
    c for c in df.columns
    if c.startswith("clust_") and (
        c.endswith("_count") or c.endswith("_impact_sum")
    )
]

# (3) 뉴스 임베딩 64차원
news_emb_cols = [c for c in df.columns if c.startswith("news_emb_")]

# (4) 범주형(코드) 피처
cat_cols = [
    "spread_regime_code",
    "crude_vs5_regime_code",
    "spread_extreme",
    "crude_vs5_extreme",
]

# 실제 존재하는 컬럼만 필터링
num_cols = [c for c in (num_cols_base + clust_num_cols + news_emb_cols) if c in df.columns]
cat_cols = [c for c in cat_cols if c in df.columns]

print("num_cols:", len(num_cols))
print("cat_cols:", cat_cols)

# 최종 사용 피처
feature_cols = num_cols + cat_cols


num_cols: 103
cat_cols: ['spread_regime_code', 'crude_vs5_regime_code', 'spread_extreme', 'crude_vs5_extreme']


In [10]:
from sklearn.preprocessing import StandardScaler

# Date를 인덱스로
df_model = df.set_index("Date").copy()

# 타깃/피처가 모두 있는 행만 사용
used_cols = feature_cols + ["y_ret"]
df_model = df_model[used_cols].dropna().copy()

# 날짜 분할 기준
train_end = pd.Timestamp("2020-12-31")
valid_end = pd.Timestamp("2022-12-31")

dates_all = df_model.index

train_mask = dates_all <= train_end
valid_mask = (dates_all > train_end) & (dates_all <= valid_end)
test_mask  = dates_all > valid_end

# 스케일러는 numeric feature만, train 구간으로 fitting
scaler = StandardScaler()
scaler.fit(df_model.loc[train_mask, num_cols])

# 전체에 변환 적용
df_model[num_cols] = scaler.transform(df_model[num_cols])
# cat_cols는 그대로 사용 (int/코드형)


In [11]:
def build_sequences(df_model, feature_cols, target_col="y_ret", window=60):
    """
    df_model: Date index, feature_cols + target_col 포함
    window: 시퀀스 길이 (L)
    """
    X_list = []
    y_list = []
    date_list = []
    
    values = df_model[feature_cols].values
    targets = df_model[target_col].values
    dates = df_model.index.values
    
    N = len(df_model)
    
    for i in range(window, N):
        X_list.append(values[i-window:i])   # [window, d_feat]
        y_list.append(targets[i])           # y_t (이미 shift로 y_{t} = ret_{t} or ret_{t+1} 맞춰둠)
        date_list.append(dates[i])
    
    X = np.asarray(X_list, dtype=np.float32)
    y = np.asarray(y_list, dtype=np.float32)
    date_arr = np.asarray(date_list)
    
    return X, y, date_arr

WINDOW = 60
X, y, dates_seq = build_sequences(df_model, feature_cols, target_col="y_ret", window=WINDOW)

print("X shape:", X.shape)  # [N_seq, L, d_feat]
print("y shape:", y.shape)


X shape: (2924, 60, 107)
y shape: (2924,)


In [12]:
train_mask_seq = dates_seq <= train_end
valid_mask_seq = (dates_seq > train_end) & (dates_seq <= valid_end)
test_mask_seq  = dates_seq > valid_end

X_train, y_train = X[train_mask_seq], y[train_mask_seq]
X_valid, y_valid = X[valid_mask_seq], y[valid_mask_seq]
X_test,  y_test  = X[test_mask_seq],  y[test_mask_seq]

print(X_train.shape, X_valid.shape, X_test.shape)


(1699, 60, 107) (503, 60, 107) (722, 60, 107)


In [13]:
import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

class SequenceDataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.from_numpy(X)  # [N, L, d]
        self.y = torch.from_numpy(y)  # [N]
    def __len__(self):
        return len(self.X)
    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]

train_ds = SequenceDataset(X_train, y_train)
valid_ds = SequenceDataset(X_valid, y_valid)
test_ds  = SequenceDataset(X_test,  y_test)

train_loader = DataLoader(train_ds, batch_size=64, shuffle=True)
valid_loader = DataLoader(valid_ds, batch_size=256, shuffle=False)
test_loader  = DataLoader(test_ds,  batch_size=256, shuffle=False)

# ---------------------------
# GRU 기반 회귀 모델
# ---------------------------
class GRURegressor(nn.Module):
    def __init__(self, input_dim, hidden_dim=64, num_layers=2, dropout=0.1):
        super().__init__()
        self.gru = nn.GRU(
            input_size=input_dim,
            hidden_size=hidden_dim,
            num_layers=num_layers,
            dropout=dropout if num_layers > 1 else 0.0,
            batch_first=True,
        )
        self.fc = nn.Sequential(
            nn.Linear(hidden_dim, hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, 1),
        )
        
    def forward(self, x):
        # x: [B, L, D]
        out, h_n = self.gru(x)      # out: [B, L, H]
        h_last = out[:, -1, :]      # 마지막 타임스텝
        y_hat = self.fc(h_last).squeeze(-1)  # [B]
        return y_hat

input_dim = X_train.shape[-1]
model = GRURegressor(input_dim=input_dim, hidden_dim=64, num_layers=2, dropout=0.1).to(device)


In [14]:
from sklearn.metrics import mean_squared_error, mean_absolute_error

def evaluate(model, loader):
    model.eval()
    ys = []
    preds = []
    with torch.no_grad():
        for xb, yb in loader:
            xb = xb.to(device)
            yb = yb.to(device)
            y_hat = model(xb)
            ys.append(yb.cpu().numpy())
            preds.append(y_hat.cpu().numpy())
    ys = np.concatenate(ys)
    preds = np.concatenate(preds)
    
    mse = mean_squared_error(ys, preds)
    mae = mean_absolute_error(ys, preds)
    # 방향성 정확도 (Up/Down)
    direction_acc = ((np.sign(ys) == np.sign(preds)).astype(float)).mean()
    return mse, mae, direction_acc, ys, preds

def train_model(model, train_loader, valid_loader, epochs=30, lr=1e-3):
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    criterion = nn.MSELoss()
    
    best_val = float("inf")
    best_state = None
    
    for epoch in range(1, epochs+1):
        model.train()
        train_losses = []
        for xb, yb in train_loader:
            xb = xb.to(device)
            yb = yb.to(device)
            
            optimizer.zero_grad()
            y_hat = model(xb)
            loss = criterion(y_hat, yb)
            loss.backward()
            optimizer.step()
            
            train_losses.append(loss.item())
        
        train_loss = np.mean(train_losses)
        val_mse, val_mae, val_dir, _, _ = evaluate(model, valid_loader)
        
        print(f"[Epoch {epoch:03d}] "
              f"train_loss={train_loss:.5f} "
              f"val_mse={val_mse:.5f} val_mae={val_mae:.5f} "
              f"val_dirAcc={val_dir:.3f}")
        
        if val_mse < best_val:
            best_val = val_mse
            best_state = model.state_dict()
    
    if best_state is not None:
        model.load_state_dict(best_state)
    return model

# 학습
model = train_model(model, train_loader, valid_loader, epochs=30, lr=1e-3)

# 평가
val_mse, val_mae, val_dir, yv, pv = evaluate(model, valid_loader)
test_mse, test_mae, test_dir, yt, pt = evaluate(model, test_loader)

print("\n[VALID] MSE:", val_mse, "MAE:", val_mae, "DirAcc:", val_dir)
print("[TEST ] MSE:", test_mse, "MAE:", test_mae, "DirAcc:", test_dir)


[Epoch 001] train_loss=0.00431 val_mse=0.00181 val_mae=0.03640 val_dirAcc=0.427
[Epoch 002] train_loss=0.00114 val_mse=0.00099 val_mae=0.02494 val_dirAcc=0.435
[Epoch 003] train_loss=0.00090 val_mse=0.00082 val_mae=0.02211 val_dirAcc=0.469
[Epoch 004] train_loss=0.00084 val_mse=0.00083 val_mae=0.02221 val_dirAcc=0.479
[Epoch 005] train_loss=0.00077 val_mse=0.00080 val_mae=0.02174 val_dirAcc=0.475
[Epoch 006] train_loss=0.00071 val_mse=0.00078 val_mae=0.02142 val_dirAcc=0.485
[Epoch 007] train_loss=0.00067 val_mse=0.00086 val_mae=0.02278 val_dirAcc=0.455
[Epoch 008] train_loss=0.00067 val_mse=0.00084 val_mae=0.02244 val_dirAcc=0.459
[Epoch 009] train_loss=0.00061 val_mse=0.00077 val_mae=0.02111 val_dirAcc=0.499
[Epoch 010] train_loss=0.00059 val_mse=0.00083 val_mae=0.02255 val_dirAcc=0.453
[Epoch 011] train_loss=0.00058 val_mse=0.00081 val_mae=0.02200 val_dirAcc=0.447
[Epoch 012] train_loss=0.00055 val_mse=0.00083 val_mae=0.02242 val_dirAcc=0.465
[Epoch 013] train_loss=0.00053 val_mse=0

In [15]:
# 예: test 구간에서 종가 복원 (yt, pt는 y_{t} = ret_t)
# dates_seq[test_mask_seq]와 df의 brent_close를 맞춰서 close_t를 가져와야 합니다.

test_dates = dates_seq[test_mask_seq]
close_series = df.set_index("Date")["brent_close"]

close_t = close_series.reindex(test_dates)  # t 시점 종가
price_pred_next = close_t.values * np.exp(pt)  # log 수익률 기준이면
price_actual_next = close_t.values * np.exp(yt)


In [16]:
# y_train, y_valid, y_test는 이전 코드에서 이미 있음

import numpy as np
from sklearn.metrics import mean_squared_error, mean_absolute_error

def dir_acc(y_true, y_pred):
    return (np.sign(y_true) == np.sign(y_pred)).astype(float).mean()

def print_baselines(y_train, y_valid, y_test):
    # 1) 항상 0 예측 (수익률=0)
    zero_pred = 0.0
    for split, y in [("TRAIN", y_train), ("VALID", y_valid), ("TEST", y_test)]:
        mse = mean_squared_error(y, np.full_like(y, zero_pred))
        mae = mean_absolute_error(y, np.full_like(y, zero_pred))
        da  = dir_acc(y, np.full_like(y, zero_pred))
        print(f"[ZERO-{split}] MSE={mse:.6f} MAE={mae:.6f} DirAcc={da:.3f}")
    
    # 2) naive: 내일 수익률 = 오늘 수익률 (y_t+1 ≈ y_t)
    #   → sequence에서 마지막 타임스텝의 타깃을 그대로 예측한다고 가정
    #   (y 배열을 한 칸 shift 해서 비교)
    def naive_from_series(y):
        # y_{t+1}를 예측한다고 보면, naive 예측은 y_t
        return y[:-1], y[1:]
    
    for split, y in [("TRAIN", y_train), ("VALID", y_valid), ("TEST", y_test)]:
        y_naive_pred, y_true = naive_from_series(y)
        mse = mean_squared_error(y_true, y_naive_pred)
        mae = mean_absolute_error(y_true, y_naive_pred)
        da  = dir_acc(y_true, y_naive_pred)
        print(f"[NAIVE-{split}] MSE={mse:.6f} MAE={mae:.6f} DirAcc={da:.3f}")

print_baselines(y_train, y_valid, y_test)


[ZERO-TRAIN] MSE=0.000687 MAE=0.016864 DirAcc=0.005
[ZERO-VALID] MSE=0.000631 MAE=0.018541 DirAcc=0.004
[ZERO-TEST] MSE=0.000339 MAE=0.013974 DirAcc=0.003
[NAIVE-TRAIN] MSE=0.001378 MAE=0.024898 DirAcc=0.478
[NAIVE-VALID] MSE=0.001230 MAE=0.025895 DirAcc=0.512
[NAIVE-TEST] MSE=0.000658 MAE=0.020137 DirAcc=0.477


# Multi-Horizon GRU Baseline (5일 / 10일 / 20일)

In [17]:
# Multi-horizon target
df["ret_5d"]  = np.log(df["brent_close"].shift(-5)) - np.log(df["brent_close"])
df["ret_10d"] = np.log(df["brent_close"].shift(-10)) - np.log(df["brent_close"])
df["ret_20d"] = np.log(df["brent_close"].shift(-20)) - np.log(df["brent_close"])

# Drop last rows with NaN target
df = df.dropna(subset=["ret_5d", "ret_10d", "ret_20d"]).reset_index(drop=True)

In [18]:
# (1) 숫자형 피처들
num_cols_base = [
    # 가격/기술
    "brent_close",
    "brent_ret_1d", "brent_ret_5d", "brent_ret_20d",
    "brent_ma_5", "brent_ma_20", "brent_ma_60",
    "brent_vol_5d", "high_low_range",
    
    # 스프레드/재고/생산/수출입
    "brent_wti_spread", "spread_chg_5d",
    "crude_stock_vs_5yr_avg", "crude_vs5_zscore",
    "prod_weekly", "prod_4w_ma", "prod_wow_change",
    "crude_exports", "crude_imports", "import_4w_ma", "net_imports",
    "refinery_run_rate",
    
    # 거시/금리/달러/ETF
    "DXY_DX-Y.NYB",
    "('DXY', 'DX-Y.NYB')_ret_1d",
    "us2y_yield", "us10y_yield", "term_spread",
    "XLE_XLE", "('XLE', 'XLE')_ret_1d",
    "VDE_VDE", "('VDE', 'VDE')_ret_1d",
    "IXC_IXC", "('IXC', 'IXC')_ret_1d",
    
    # BDI
    "bdi_price", "bdi_change_%",
    
    # CFTC shock 기반 심리
    "wti_mm_net_long_ratio_shock_decay",
    "wti_producer_hedge_ratio_shock_decay",
    "wti_mm_position_change_wow_shock_decay",
    
    # 뉴스 전체 개수
    "news_total_count",
]

# (2) 뉴스 클러스터 피처: count + impact_sum만 사용 (mean은 생략)
clust_num_cols = [
    c for c in df.columns
    if c.startswith("clust_") and (
        c.endswith("_count") or c.endswith("_impact_sum")
    )
]

# (3) 뉴스 임베딩 64차원
news_emb_cols = [c for c in df.columns if c.startswith("news_emb_")]

# (4) 범주형(코드) 피처
cat_cols = [
    "spread_regime_code",
    "crude_vs5_regime_code",
    "spread_extreme",
    "crude_vs5_extreme",
]

# 실제 존재하는 컬럼만 필터링
num_cols = [c for c in (num_cols_base + clust_num_cols + news_emb_cols) if c in df.columns]
cat_cols = [c for c in cat_cols if c in df.columns]

print("num_cols:", len(num_cols))
print("cat_cols:", cat_cols)

# 최종 사용 피처
feature_cols = num_cols + cat_cols

num_cols: 103
cat_cols: ['spread_regime_code', 'crude_vs5_regime_code', 'spread_extreme', 'crude_vs5_extreme']


In [19]:
SEQ_LEN = 30  # 30일 윈도우

def build_sequences(df, num_features, cat_features, seq_len=SEQ_LEN):
    X_num, X_cat = [], []
    y = []

    num_vals = df[num_features].values
    cat_vals = df[cat_features].values if len(cat_features) > 0 else None

    y_vals = df[["ret_5d", "ret_10d", "ret_20d"]].values

    for i in range(len(df) - seq_len):
        X_num.append(num_vals[i:i+seq_len])
        if cat_vals is not None:
            X_cat.append(cat_vals[i])  # categorical은 sequence 아님 → 오늘 값만 사용
        y.append(y_vals[i+seq_len])

    X_num = np.array(X_num)
    y = np.array(y)
    if len(cat_features) > 0:
        X_cat = np.array(X_cat)
        return X_num, X_cat, y
    else:
        return X_num, None, y


In [20]:
train_ratio = 0.7
valid_ratio = 0.15

N = len(df)
train_end = int(N * train_ratio)
valid_end = int(N * (train_ratio + valid_ratio))

train_df = df.iloc[:train_end]
valid_df = df.iloc[train_end:valid_end]
test_df  = df.iloc[valid_end:]

Xnum_tr, Xcat_tr, y_tr = build_sequences(train_df, num_cols, cat_cols)
Xnum_va, Xcat_va, y_va = build_sequences(valid_df, num_cols, cat_cols)
Xnum_te, Xcat_te, y_te = build_sequences(test_df,  num_cols, cat_cols)


In [26]:
import tensorflow as tf
from tensorflow.keras import layers, models

num_dim = Xnum_tr.shape[-1]
cat_dim = (Xcat_tr.shape[-1] if Xcat_tr is not None else 0)

num_input = layers.Input(shape=(SEQ_LEN, num_dim))

# Numeric GRU backbone
x = layers.GRU(64, return_sequences=False)(num_input)
x = layers.Dense(64, activation="relu")(x)

# Optional categorical input
if cat_dim > 0:
    cat_input = layers.Input(shape=(cat_dim,))
    cat_x = layers.Dense(16, activation="relu")(cat_input)
    hidden = layers.Concatenate()([x, cat_x])
    inputs = [num_input, cat_input]
else:
    hidden = x
    inputs = num_input

# Multi-head output (5d, 10d, 20d)
out = layers.Dense(3, activation="linear")(hidden)

model = models.Model(inputs=inputs, outputs=out)
model.compile(loss=tf.keras.losses.Huber(), optimizer="adam")
model.summary()


Model: "functional_1"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer_2       │ (None, 30, 103)   │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ gru_1 (GRU)         │ (None, 64)        │     32,448 │ input_layer_2[0]… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ input_layer_3       │ (None, 4)         │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_3 (Dense)     │ (None, 64)        │      4,160 │ gru_1[0][0]       │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_4 (Dense)     │ (None, 16)        │         80 │ input_layer_3[0]… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ concatenate_1       │ (None, 80)        │          0 │ dense_3[0][0],    │
│ (Concatenate)       │                   │            │ dense_4[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_5 (Dense)     │ (None, 3)         │        243 │ concatenate_1[0]… │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 36,931 (144.26 KB)

 Trainable params: 36,931 (144.26 KB)

 Non-trainable params: 0 (0.00 B)

In [27]:
if Xcat_tr is None:
    history = model.fit(
        Xnum_tr, y_tr,
        validation_data=(Xnum_va, y_va),
        epochs=40,
        batch_size=32
    )
else:
    history = model.fit(
        [Xnum_tr, Xcat_tr], y_tr,
        validation_data=([Xnum_va, Xcat_va], y_va),
        epochs=40,
        batch_size=32
    )


Epoch 1/40
64/64 ━━━━━━━━━━━━━━━━━━━━ 4s 20ms/step - loss: 0.0820 - val_loss: 0.0238
Epoch 2/40
64/64 ━━━━━━━━━━━━━━━━━━━━ 1s 14ms/step - loss: 0.0135 - val_loss: 0.0176
Epoch 3/40
64/64 ━━━━━━━━━━━━━━━━━━━━ 1s 14ms/step - loss: 0.0093 - val_loss: 0.0142
Epoch 4/40
64/64 ━━━━━━━━━━━━━━━━━━━━ 1s 14ms/step - loss: 0.0077 - val_loss: 0.0131
Epoch 5/40
64/64 ━━━━━━━━━━━━━━━━━━━━ 1s 14ms/step - loss: 0.0064 - val_loss: 0.0119
Epoch 6/40
64/64 ━━━━━━━━━━━━━━━━━━━━ 1s 16ms/step - loss: 0.0058 - val_loss: 0.0149
Epoch 7/40
64/64 ━━━━━━━━━━━━━━━━━━━━ 1s 14ms/step - loss: 0.0053 - val_loss: 0.0123
Epoch 8/40
64/64 ━━━━━━━━━━━━━━━━━━━━ 1s 17ms/step - loss: 0.0049 - val_loss: 0.0128
Epoch 9/40
64/64 ━━━━━━━━━━━━━━━━━━━━ 1s 20ms/step - loss: 0.0046 - val_loss: 0.0166
Epoch 10/40
64/64 ━━━━━━━━━━━━━━━━━━━━ 1s 16ms/step - loss: 0.0044 - val_loss: 0.0150
Epoch 11/40
64/64 ━━━━━━━━━━━━━━━━━━━━ 1s 18ms/step - loss: 0.0042 - val_loss: 0.0110
Epoch 12/40
64/64 ━━━━━━━━━━━━━━━━━━━━ 1s 20ms/step - loss: 0.0

In [28]:
if Xcat_te is None:
    pred = model.predict(Xnum_te)
else:
    pred = model.predict([Xnum_te, Xcat_te])

# pred.shape = (N, 3)  # ret_5d, ret_10d, ret_20d

# 마지막 day 기준 가격 복원
close_te = test_df["brent_close"].values[SEQ_LEN:]  # sequence offset 보정

pred_price_5d  = close_te * np.exp(pred[:, 0])
pred_price_10d = close_te * np.exp(pred[:, 1])
pred_price_20d = close_te * np.exp(pred[:, 2])


13/13 ━━━━━━━━━━━━━━━━━━━━ 1s 30ms/step


In [29]:
import numpy as np

def evaluate_multi(pred_ret, y_true_ret, close_last):
    """
    pred_ret: (N,3) 예측 로그수익률 (5/10/20일)
    y_true_ret: (N,3) 실제 로그수익률
    close_last: (N,) horizon=0 시점 close price
    """
    horizons = ["5d","10d","20d"]
    results = {}

    for i, h in enumerate(horizons):
        pred_r = pred_ret[:, i]
        true_r = y_true_ret[:, i]

        mse  = np.mean((pred_r - true_r)**2)
        mae  = np.mean(np.abs(pred_r - true_r))
        diracc = np.mean(np.sign(pred_r) == np.sign(true_r))

        # 가격 복원
        pred_p = close_last * np.exp(pred_r)
        true_p = close_last * np.exp(true_r)

        price_mse = np.mean((pred_p - true_p)**2)
        price_mae = np.mean(np.abs(pred_p - true_p))

        results[h] = {
            "ret_MSE": mse,
            "ret_MAE": mae,
            "ret_DirAcc": diracc,
            "price_MSE": price_mse,
            "price_MAE": price_mae
        }

    return results

results = evaluate_multi(pred, y_te, close_te)

for h, res in results.items():
    print(f"\n===== {h} =====")
    for k, v in res.items():
        print(f"{k}: {v:.6f}")



===== 5d =====
ret_MSE: 0.025828
ret_MAE: 0.130127
ret_DirAcc: 0.467470
price_MSE: 119.659433
price_MAE: 9.039101

===== 10d =====
ret_MSE: 0.054809
ret_MAE: 0.207837
ret_DirAcc: 0.493976
price_MSE: 239.845681
price_MAE: 13.846430

===== 20d =====
ret_MSE: 0.068841
ret_MAE: 0.207018
ret_DirAcc: 0.522892
price_MSE: 271.779990
price_MAE: 13.372380


In [30]:
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.feature_selection import mutual_info_regression
from sklearn.metrics import mean_squared_error, mean_absolute_error
from sklearn.preprocessing import StandardScaler

from lightgbm import LGBMRegressor

import shap


In [31]:
# Date 정렬
df = df.sort_values("Date").reset_index(drop=True)

target_cols = ["ret_5d", "ret_10d", "ret_20d"]  # 타깃 이름 맞게 수정 가능

# 전체 데이터 길이 기준 단순 비율 split 예시 (60% / 20% / 20%)
N = len(df)
n_train = int(N * 0.6)
n_valid = int(N * 0.2)

df_train = df.iloc[:n_train].copy()
df_valid = df.iloc[n_train:n_train+n_valid].copy()
df_test  = df.iloc[n_train+n_valid:].copy()

print(len(df_train), len(df_valid), len(df_test))


1779 593 593


In [32]:
# 이미 갖고 있는 피처 리스트 사용
# feature_cols = num_cols + cat_cols
# cat_cols, num_cols는 기존 정의 사용

X_train = df_train[feature_cols].copy()
X_valid = df_valid[feature_cols].copy()
X_test  = df_test[feature_cols].copy()

y_train = df_train[target_cols].copy()
y_valid = df_valid[target_cols].copy()
y_test  = df_test[target_cols].copy()

# LightGBM용 범주형 dtype 지정
for c in cat_cols:
    if c in X_train.columns:
        X_train[c] = X_train[c].astype("category")
        X_valid[c] = X_valid[c].astype("category")
        X_test[c]  = X_test[c].astype("category")


In [33]:
def compute_mi_importance(X, y, cat_cols=None, top_k=30):
    """
    X: DataFrame (train)
    y: Series (train target 하나)
    cat_cols: 범주형 컬럼 리스트
    """
    if cat_cols is None:
        cat_cols = []

    # One-hot 인코딩 (MI 계산 전용)
    X_enc = pd.get_dummies(X, columns=cat_cols, drop_first=True)
    
    mi = mutual_info_regression(X_enc, y, random_state=42)
    mi_series = pd.Series(mi, index=X_enc.columns).sort_values(ascending=False)

    print(f"\n[MI 상위 {top_k}개]")
    print(mi_series.head(top_k))
    return mi_series

mi_results = {}  # horizon별 MI 결과 저장

for i, tgt in enumerate(target_cols):
    print(f"\n===== MI for target: {tgt} =====")
    mi_series = compute_mi_importance(X_train, y_train[tgt], cat_cols=cat_cols, top_k=30)
    mi_results[tgt] = mi_series



===== MI for target: ret_5d =====

[MI 상위 30개]
prod_weekly                               0.187193
brent_ma_20                               0.180192
VDE_VDE                                   0.176787
DXY_DX-Y.NYB                              0.173085
brent_close                               0.167468
brent_ma_5                                0.161193
us10y_yield                               0.155891
us2y_yield                                0.153909
IXC_IXC                                   0.152418
XLE_XLE                                   0.150171
crude_exports                             0.149732
prod_4w_ma                                0.147478
brent_ma_60                               0.141821
bdi_price                                 0.138868
import_4w_ma                              0.132711
term_spread                               0.132316
crude_stock_vs_5yr_avg                    0.123374
crude_vs5_zscore                          0.122774
refinery_run_rate                 

In [39]:
from lightgbm import LGBMRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error
import numpy as np
import pandas as pd

def train_lgb_and_importance(X_tr, y_tr, X_va, y_va, cat_cols=None, target_name="ret_5d"):
    params = {
        "n_estimators": 500,
        "learning_rate": 0.05,
        "max_depth": -1,
        "num_leaves": 31,
        "subsample": 0.8,
        "colsample_bytree": 0.8,
        "random_state": 42,
        "n_jobs": -1,
    }
    model = LGBMRegressor(**params)

    # --- 1) 카테고리컬 컬럼 → 인덱스로 변환 ---
    cat_features = []
    if cat_cols is not None:
        for c in cat_cols:
            if c in X_tr.columns:
                cat_features.append(X_tr.columns.get_loc(c))

    # --- 2) DataFrame → numpy 로 변환해서 학습 (feature name 안 씀) ---
    X_tr_np = X_tr.to_numpy()
    X_va_np = X_va.to_numpy()

    model.fit(
        X_tr_np, y_tr,
        eval_set=[(X_va_np, y_va)],
        eval_metric="l2",
        categorical_feature=cat_features if len(cat_features) > 0 else None,
    )

    # 성능 확인
    pred_va = model.predict(X_va_np)
    mse = mean_squared_error(y_va, pred_va)
    mae = mean_absolute_error(y_va, pred_va)
    print(f"[{target_name}] valid MSE={mse:.6f}, MAE={mae:.6f}")

    # 중요도 (gain 기준) – 순서는 여전히 X_tr.columns 기준으로 가능
    fi = pd.Series(model.feature_importances_, index=X_tr.columns)
    fi = fi.sort_values(ascending=False)

    print(f"\n[LightGBM Feature Importance: {target_name} 상위 30개]")
    print(fi.head(30))
    return model, fi


lgb_models = {}
lgb_importances = {}

for tgt in target_cols:
    print(f"\n===== LightGBM for target: {tgt} =====")
    model, fi = train_lgb_and_importance(
        X_train, y_train[tgt],
        X_valid, y_valid[tgt],
        cat_cols=cat_cols,
        target_name=tgt
    )
    lgb_models[tgt] = model
    lgb_importances[tgt] = fi



===== LightGBM for target: ret_5d =====
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.003154 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 25509
[LightGBM] [Info] Number of data points in the train set: 1779, number of used features: 107
[LightGBM] [Info] Start training from score -0.001681
[ret_5d] valid MSE=0.004901, MAE=0.053391

[LightGBM Feature Importance: ret_5d 상위 30개]
high_low_range                            525
brent_ret_5d                              517
brent_close                               472
brent_vol_5d                              428
brent_ret_20d                             419
brent_ma_20                               414
brent_ret_1d                              402
brent_ma_5                                398
bdi_price                                 379
crude_exports                             370
crude_imports                             369
brent_wti_spread     

In [40]:
def compute_shap_importance(model, X, top_k=30, target_name="ret_5d", sample_n=1000):
    # SHAP 계산 비용 줄이기 위해 샘플링
    if len(X) > sample_n:
        X_sample = X.sample(sample_n, random_state=42)
    else:
        X_sample = X

    explainer = shap.TreeExplainer(model)
    shap_values = explainer.shap_values(X_sample)

    # 회귀 모델이라 shap_values shape = (n_sample, n_features)
    shap_abs_mean = np.abs(shap_values).mean(axis=0)
    shap_importance = pd.Series(shap_abs_mean, index=X_sample.columns).sort_values(ascending=False)

    print(f"\n[SHAP Importance: {target_name} 상위 {top_k}개]")
    print(shap_importance.head(top_k))

    return shap_importance, shap_values, X_sample

shap_importances = {}

for tgt in target_cols:
    print(f"\n===== SHAP for target: {tgt} =====")
    model = lgb_models[tgt]
    shap_imp, shap_vals, X_sample = compute_shap_importance(
        model, X_valid, top_k=30, target_name=tgt, sample_n=1000
    )
    shap_importances[tgt] = shap_imp

    # 원하면 summary_plot도 가능 (노트북 환경에서)
    # shap.summary_plot(shap_vals, X_sample)



===== SHAP for target: ret_5d =====

[SHAP Importance: ret_5d 상위 30개]
crude_exports             0.014963
brent_ma_60               0.007472
DXY_DX-Y.NYB              0.005665
term_spread               0.004363
brent_ret_1d              0.003630
crude_stock_vs_5yr_avg    0.003625
brent_close               0.003432
us2y_yield                0.003403
brent_ma_5                0.003143
net_imports               0.002374
prod_4w_ma                0.002259
bdi_price                 0.002042
XLE_XLE                   0.002003
brent_wti_spread          0.001917
brent_ret_5d              0.001862
brent_ret_20d             0.001827
VDE_VDE                   0.001806
crude_imports             0.001798
('IXC', 'IXC')_ret_1d     0.001744
import_4w_ma              0.001688
us10y_yield               0.001418
prod_wow_change           0.001359
brent_ma_20               0.001358
high_low_range            0.001293
refinery_run_rate         0.001268
brent_vol_5d              0.001211
prod_weekly        

In [41]:
def aggregate_importance(mi_series, lgb_series, shap_series, top_k=40):
    # 공통 피처 기준으로 합치기
    common_idx = mi_series.index.intersection(lgb_series.index).intersection(shap_series.index)

    df_imp = pd.DataFrame({
        "mi": mi_series.reindex(common_idx),
        "lgb": lgb_series.reindex(common_idx),
        "shap": shap_series.reindex(common_idx),
    })

    # 각 중요도를 내림차순 기준 rank로 변환 (작을수록 더 중요)
    df_imp["mi_rank"]   = df_imp["mi"].rank(ascending=False)
    df_imp["lgb_rank"]  = df_imp["lgb"].rank(ascending=False)
    df_imp["shap_rank"] = df_imp["shap"].rank(ascending=False)

    df_imp["avg_rank"] = df_imp[["mi_rank","lgb_rank","shap_rank"]].mean(axis=1)
    df_imp = df_imp.sort_values("avg_rank")

    print("\n[통합 중요도 순위 상위 피처]")
    return df_imp.head(top_k)

agg_results = {}

for tgt in target_cols:
    print(f"\n===== Aggregated Importance for target: {tgt} =====")
    mi_s   = mi_results[tgt]
    lgb_s  = lgb_importances[tgt]
    shap_s = shap_importances[tgt]

    agg = aggregate_importance(mi_s, lgb_s, shap_s, top_k=40)
    agg_results[tgt] = agg
    print(agg)



===== Aggregated Importance for target: ret_5d =====

[통합 중요도 순위 상위 피처]
                                              mi  lgb      shap  mi_rank  \
brent_close                             0.167468  472  0.003432      5.0   
crude_exports                           0.149732  370  0.014963     11.0   
brent_ma_5                              0.161193  398  0.003143      6.0   
DXY_DX-Y.NYB                            0.173085  249  0.005665      4.0   
brent_ma_20                             0.180192  414  0.001358      2.0   
bdi_price                               0.138868  379  0.002042     14.0   
VDE_VDE                                 0.176787  253  0.001806      3.0   
XLE_XLE                                 0.150171  267  0.002003     10.0   
brent_ret_5d                            0.061489  517  0.001862     24.0   
us10y_yield                             0.155891  277  0.001418      7.0   
brent_ret_20d                           0.071092  419  0.001827     23.0   
prod_4w_ma     

---

In [42]:
core_num_features = [
    # ① 브렌트 레벨 / 추세 / 변동성
    "brent_close",
    "brent_ma_5", "brent_ma_20", "brent_ma_60",
    "brent_ret_5d", "brent_ret_20d",
    "brent_vol_5d", "high_low_range",

    # ② 스프레드 / 재고 / 생산 / 수출입
    "brent_wti_spread", "spread_chg_5d",
    "crude_stock_vs_5yr_avg", "crude_vs5_zscore",
    "prod_weekly", "prod_4w_ma", "prod_wow_change",
    "crude_exports", "crude_imports",
    "import_4w_ma", "net_imports",
    "refinery_run_rate",

    # ③ 달러 / 금리 / ETF
    "DXY_DX-Y.NYB",
    "us2y_yield", "us10y_yield", "term_spread",
    "XLE_XLE", "VDE_VDE", "IXC_IXC",

    # ④ BDI
    "bdi_price", "bdi_change_%",

    # ⑤ CFTC 행동(심리) shock-decay
    "wti_mm_net_long_ratio_shock_decay",
    "wti_producer_hedge_ratio_shock_decay",
    "wti_mm_position_change_wow_shock_decay",

    # ⑥ 뉴스 임베딩 – 상위 몇 개만 뽑기 (예시)
    "news_emb_22", "news_emb_29", "news_emb_38",
    "news_emb_52", "news_emb_56", "news_emb_61", "news_emb_63",
]
core_num_features = [c for c in core_num_features if c in df.columns]


In [43]:
core_cat_features = [
    "spread_regime_code",
    "crude_vs5_regime_code",
]
core_cat_features = [c for c in core_cat_features if c in df.columns]


In [44]:
SEQ_LEN = 20  # 예: 20일 히스토리

use_cols = core_num_features + core_cat_features

data = df.dropna(subset=use_cols + target_cols).copy()

# 인덱스 정렬
data = data.sort_values("Date").reset_index(drop=True)

# -----------------------------
# 1) 숫자 / 카테고리 분리
# -----------------------------
num_cols = core_num_features
cat_cols = core_cat_features

X_num = data[num_cols].values.astype("float32")

if len(cat_cols) > 0:
    X_cat = data[cat_cols].values.astype("float32")
else:
    X_cat = None

y_all = data[["ret_5d", "ret_10d", "ret_20d"]].values.astype("float32")
close_all = data["brent_close"].values.astype("float32")

# -----------------------------
# 2) 시퀀스로 변환
# -----------------------------
def make_sequence(X_num, X_cat, y, close, seq_len=20):
    Xn_seq, Xc_seq, y_seq, close_seq = [], [], [], []
    N = len(X_num)
    for i in range(N - seq_len):
        Xn_seq.append(X_num[i:i+seq_len])
        if X_cat is not None:
            Xc_seq.append(X_cat[i+seq_len-1])  # 마지막 시점의 regime만 사용
        y_seq.append(y[i+seq_len-1])
        close_seq.append(close[i+seq_len-1])
    Xn_seq = np.array(Xn_seq)
    y_seq  = np.array(y_seq)
    close_seq = np.array(close_seq)
    Xc_seq = np.array(Xc_seq) if X_cat is not None else None
    return Xn_seq, Xc_seq, y_seq, close_seq

Xnum_seq, Xcat_seq, y_seq, close_seq = make_sequence(
    X_num, X_cat, y_all, close_all, seq_len=SEQ_LEN
)


In [46]:
import numpy as np
import pandas as pd

from sklearn.metrics import mean_squared_error, mean_absolute_error

import tensorflow as tf
from tensorflow.keras import layers, models

# --------------------------------------------------------------------
# 0. 재현성 세팅 (선택)
# --------------------------------------------------------------------
SEED = 42
np.random.seed(SEED)
tf.random.set_seed(SEED)

# --------------------------------------------------------------------
# 1. 타겟(ret_5d / ret_10d / ret_20d) 준비
#    - 이미 있으면 사용, 없으면 생성
# --------------------------------------------------------------------
if "ret_5d" not in df.columns or "ret_10d" not in df.columns or "ret_20d" not in df.columns:
    df["ret_5d"]  = np.log(df["brent_close"].shift(-5))  - np.log(df["brent_close"])
    df["ret_10d"] = np.log(df["brent_close"].shift(-10)) - np.log(df["brent_close"])
    df["ret_20d"] = np.log(df["brent_close"].shift(-20)) - np.log(df["brent_close"])

target_cols = ["ret_5d", "ret_10d", "ret_20d"]

# --------------------------------------------------------------------
# 2. 코어 피처 선택 (MI + LGB + SHAP 결과 반영)
# --------------------------------------------------------------------
core_num_features = [
    # ① 브렌트 레벨 / 추세 / 변동성
    "brent_close",
    "brent_ma_5", "brent_ma_20", "brent_ma_60",
    "brent_ret_5d", "brent_ret_20d",
    "brent_vol_5d", "high_low_range",

    # ② 스프레드 / 재고 / 생산 / 수출입
    "brent_wti_spread", "spread_chg_5d",
    "crude_stock_vs_5yr_avg", "crude_vs5_zscore",
    "prod_weekly", "prod_4w_ma", "prod_wow_change",
    "crude_exports", "crude_imports",
    "import_4w_ma", "net_imports",
    "refinery_run_rate",

    # ③ 달러 / 금리 / ETF
    "DXY_DX-Y.NYB",
    "us2y_yield", "us10y_yield", "term_spread",
    "XLE_XLE", "VDE_VDE", "IXC_IXC",

    # ④ BDI
    "bdi_price", "bdi_change_%",

    # ⑤ CFTC 행동(심리) shock-decay
    "wti_mm_net_long_ratio_shock_decay",
    "wti_producer_hedge_ratio_shock_decay",
    "wti_mm_position_change_wow_shock_decay",

    # ⑥ 뉴스 임베딩 – 중요도 상위 일부
    "news_emb_22", "news_emb_29", "news_emb_38",
    "news_emb_52", "news_emb_56", "news_emb_61", "news_emb_63",

    "clust_id"
]

core_cat_features = [
    "spread_regime_code",
    "crude_vs5_regime_code",
]

# 실제 존재하는 컬럼만 필터링
core_num_features = [c for c in core_num_features if c in df.columns]
core_cat_features = [c for c in core_cat_features if c in df.columns]

print("[NUM FEATURES]", len(core_num_features), core_num_features)
print("[CAT FEATURES]", len(core_cat_features), core_cat_features)

# --------------------------------------------------------------------
# 3. 결측치 제거 + 정렬
# --------------------------------------------------------------------
use_cols = core_num_features + core_cat_features + target_cols + ["brent_close", "Date"]
use_cols = [c for c in use_cols if c in df.columns]

data = df[use_cols].copy()
data = data.sort_values("Date").reset_index(drop=True)
data = data.dropna(subset=core_num_features + target_cols).reset_index(drop=True)

print("데이터 shape:", data.shape)

# --------------------------------------------------------------------
# 4. 넘파이 배열로 변환
# --------------------------------------------------------------------
num_cols = core_num_features
cat_cols = core_cat_features

X_num_all = data[num_cols].values.astype("float32")
X_cat_all = data[cat_cols].values.astype("float32") if len(cat_cols) > 0 else None

y_all     = data[target_cols].values.astype("float32")
close_all = data["brent_close"].values.astype("float32")

# --------------------------------------------------------------------
# 5. 시퀀스 생성 (SEQ_LEN일 히스토리 → 그날의 ret_5d/10d/20d)
# --------------------------------------------------------------------
SEQ_LEN = 20  # 필요하면 10, 30 등 바꿔서 실험

def make_sequence(X_num, X_cat, y, close, seq_len=20):
    Xn_seq, Xc_seq, y_seq, close_seq = [], [], [], []
    N = len(X_num)
    for i in range(N - seq_len):
        Xn_seq.append(X_num[i : i+seq_len])
        if X_cat is not None:
            # 카테고리 피처는 시퀀스 마지막 날(현재날) 것만 사용
            Xc_seq.append(X_cat[i + seq_len - 1])
        y_seq.append(y[i + seq_len - 1])
        close_seq.append(close[i + seq_len - 1])
    Xn_seq = np.array(Xn_seq)
    y_seq  = np.array(y_seq)
    close_seq = np.array(close_seq)
    Xc_seq = np.array(Xc_seq) if X_cat is not None else None
    return Xn_seq, Xc_seq, y_seq, close_seq

Xnum_seq, Xcat_seq, y_seq, close_seq = make_sequence(
    X_num_all, X_cat_all, y_all, close_all, seq_len=SEQ_LEN
)

print("Xnum_seq:", Xnum_seq.shape)
if Xcat_seq is not None:
    print("Xcat_seq:", Xcat_seq.shape)
print("y_seq:", y_seq.shape)
print("close_seq:", close_seq.shape)

# --------------------------------------------------------------------
# 6. Train / Valid / Test 시계열 Split
# --------------------------------------------------------------------
N = Xnum_seq.shape[0]
train_end = int(N * 0.7)
valid_end = int(N * 0.85)

Xnum_tr = Xnum_seq[:train_end]
Xnum_va = Xnum_seq[train_end:valid_end]
Xnum_te = Xnum_seq[valid_end:]

if Xcat_seq is not None:
    Xcat_tr = Xcat_seq[:train_end]
    Xcat_va = Xcat_seq[train_end:valid_end]
    Xcat_te = Xcat_seq[valid_end:]
else:
    Xcat_tr = Xcat_va = Xcat_te = None

y_tr = y_seq[:train_end]
y_va = y_seq[train_end:valid_end]
y_te = y_seq[valid_end:]

close_tr = close_seq[:train_end]
close_va = close_seq[train_end:valid_end]
close_te = close_seq[valid_end:]

print(f"Train: {Xnum_tr.shape[0]}, Valid: {Xnum_va.shape[0]}, Test: {Xnum_te.shape[0]}")

# --------------------------------------------------------------------
# 7. GRU 모델 정의
# --------------------------------------------------------------------
num_dim = Xnum_tr.shape[-1]
cat_dim = Xcat_tr.shape[-1] if Xcat_tr is not None else 0

num_input = layers.Input(shape=(SEQ_LEN, num_dim))

x = layers.GRU(64, return_sequences=False)(num_input)
x = layers.Dense(64, activation="relu")(x)

if cat_dim > 0:
    cat_input = layers.Input(shape=(cat_dim,))
    cat_x = layers.Dense(16, activation="relu")(cat_input)
    hidden = layers.Concatenate()([x, cat_x])
    inputs = [num_input, cat_input]
else:
    cat_input = None
    hidden = x
    inputs = num_input

# 3개 호라이즌(5d, 10d, 20d) 동시 예측
out = layers.Dense(3, activation="linear")(hidden)

model = models.Model(inputs=inputs, outputs=out)
model.compile(optimizer=tf.keras.optimizers.Adam(1e-3), loss="mse")

model.summary()

# --------------------------------------------------------------------
# 8. 학습
# --------------------------------------------------------------------
EPOCHS = 50
BATCH_SIZE = 32

if Xcat_tr is None:
    history = model.fit(
        Xnum_tr, y_tr,
        validation_data=(Xnum_va, y_va),
        epochs=EPOCHS,
        batch_size=BATCH_SIZE,
        verbose=1,
    )
else:
    history = model.fit(
        [Xnum_tr, Xcat_tr], y_tr,
        validation_data=([Xnum_va, Xcat_va], y_va),
        epochs=EPOCHS,
        batch_size=BATCH_SIZE,
        verbose=1,
    )

# --------------------------------------------------------------------
# 9. 평가 함수 (ret + price, 각 호라이즌별)
# --------------------------------------------------------------------
def eval_multi_horizon(y_true, y_pred, close_base, horizon_idx, label):
    """
    y_true, y_pred: shape (N, 3) [ret_5d, ret_10d, ret_20d]
    close_base: 그날 브렌트 종가 (shape (N,) 혹은 (N,1) 혹은 (N,2)일 수도 있어서 방어)
    horizon_idx: 0(5d), 1(10d), 2(20d)
    """
    # --- close_base shape 정리 ---
    close_base = np.asarray(close_base)
    if close_base.ndim == 2:
        # (N, 1) 또는 (N, 2)이면 첫 번째 컬럼만 사용
        close_base = close_base[:, 0]
    close_base = close_base.reshape(-1)  # (N,)

    t = y_true[:, horizon_idx]
    p = y_pred[:, horizon_idx]

    # 길이 체크 (디버깅용)
    assert len(close_base) == len(t), f"len(close_base)={len(close_base)}, len(t)={len(t)}"

    # 수익률 기준
    ret_mse = mean_squared_error(t, p)
    ret_mae = mean_absolute_error(t, p)
    dir_acc = np.mean(np.sign(t) == np.sign(p))

    # 가격 복원
    true_price = close_base * np.exp(t)
    pred_price = close_base * np.exp(p)

    price_mse = mean_squared_error(true_price, pred_price)
    price_mae = mean_absolute_error(true_price, pred_price)

    print(f"===== {label} =====")
    print(f"ret_MSE:   {ret_mse:.6f}")
    print(f"ret_MAE:   {ret_mae:.6f}")
    print(f"ret_DirAcc:{dir_acc:.6f}")
    print(f"price_MSE: {price_mse:.6f}")
    print(f"price_MAE: {price_mae:.6f}")
    print()


    print(f"===== {label} =====")
    print(f"ret_MSE:   {ret_mse:.6f}")
    print(f"ret_MAE:   {ret_mae:.6f}")
    print(f"ret_DirAcc:{dir_acc:.6f}")
    print(f"price_MSE: {price_mse:.6f}")
    print(f"price_MAE: {price_mae:.6f}")
    print()

# --------------------------------------------------------------------
# 10. 테스트셋 예측 및 결과 출력
# --------------------------------------------------------------------
if Xcat_te is None:
    pred_te = model.predict(Xnum_te)
else:
    pred_te = model.predict([Xnum_te, Xcat_te])

# horizon index: 0=5d, 1=10d, 2=20d
eval_multi_horizon(y_te, pred_te, close_te, 0, "5d")
eval_multi_horizon(y_te, pred_te, close_te, 1, "10d")
eval_multi_horizon(y_te, pred_te, close_te, 2, "20d")


[NUM FEATURES] 38 ['brent_close', 'brent_ma_5', 'brent_ma_20', 'brent_ma_60', 'brent_ret_5d', 'brent_ret_20d', 'brent_vol_5d', 'high_low_range', 'brent_wti_spread', 'spread_chg_5d', 'crude_stock_vs_5yr_avg', 'crude_vs5_zscore', 'prod_weekly', 'prod_4w_ma', 'prod_wow_change', 'crude_exports', 'crude_imports', 'import_4w_ma', 'net_imports', 'refinery_run_rate', 'us2y_yield', 'us10y_yield', 'term_spread', 'XLE_XLE', 'VDE_VDE', 'IXC_IXC', 'bdi_price', 'wti_mm_net_long_ratio_shock_decay', 'wti_producer_hedge_ratio_shock_decay', 'wti_mm_position_change_wow_shock_decay', 'news_emb_22', 'news_emb_29', 'news_emb_38', 'news_emb_52', 'news_emb_56', 'news_emb_61', 'news_emb_63', 'clust_id']
[CAT FEATURES] 2 ['spread_regime_code', 'crude_vs5_regime_code']
데이터 shape: (2965, 45)
Xnum_seq: (2945, 20, 39)
Xcat_seq: (2945, 2)
y_seq: (2945, 3)
close_seq: (2945, 2)
Train: 2061, Valid: 442, Test: 442


Model: "functional_3"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer_6       │ (None, 20, 39)    │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ gru_3 (GRU)         │ (None, 64)        │     20,160 │ input_layer_6[0]… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ input_layer_7       │ (None, 2)         │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_9 (Dense)     │ (None, 64)        │      4,160 │ gru_3[0][0]       │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_10 (Dense)    │ (None, 16)        │         48 │ input_layer_7[0]… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ concatenate_3       │ (None, 80)        │          0 │ dense_9[0][0],    │
│ (Concatenate)       │                   │            │ dense_10[0][0]    │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_11 (Dense)    │ (None, 3)         │        243 │ concatenate_3[0]… │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 24,611 (96.14 KB)

 Trainable params: 24,611 (96.14 KB)

 Non-trainable params: 0 (0.00 B)

Epoch 1/50
65/65 ━━━━━━━━━━━━━━━━━━━━ 4s 12ms/step - loss: 0.0834 - val_loss: 0.0399
Epoch 2/50
65/65 ━━━━━━━━━━━━━━━━━━━━ 1s 8ms/step - loss: 0.0283 - val_loss: 0.0292
Epoch 3/50
65/65 ━━━━━━━━━━━━━━━━━━━━ 1s 8ms/step - loss: 0.0178 - val_loss: 0.0250
Epoch 4/50
65/65 ━━━━━━━━━━━━━━━━━━━━ 1s 9ms/step - loss: 0.0146 - val_loss: 0.0207
Epoch 5/50
65/65 ━━━━━━━━━━━━━━━━━━━━ 1s 10ms/step - loss: 0.0122 - val_loss: 0.0189
Epoch 6/50
65/65 ━━━━━━━━━━━━━━━━━━━━ 1s 10ms/step - loss: 0.0110 - val_loss: 0.0180
Epoch 7/50
65/65 ━━━━━━━━━━━━━━━━━━━━ 1s 10ms/step - loss: 0.0102 - val_loss: 0.0172
Epoch 8/50
65/65 ━━━━━━━━━━━━━━━━━━━━ 1s 10ms/step - loss: 0.0090 - val_loss: 0.0175
Epoch 9/50
65/65 ━━━━━━━━━━━━━━━━━━━━ 1s 10ms/step - loss: 0.0084 - val_loss: 0.0163
Epoch 10/50
65/65 ━━━━━━━━━━━━━━━━━━━━ 1s 10ms/step - loss: 0.0084 - val_loss: 0.0162
Epoch 11/50
65/65 ━━━━━━━━━━━━━━━━━━━━ 1s 11ms/step - loss: 0.0081 - val_loss: 0.0160
Epoch 12/50
65/65 ━━━━━━━━━━━━━━━━━━━━ 1s 11ms/step - loss: 0.0078

In [48]:
df.head()

,Date,brent_close,wti_close,brent_wti_spread,brent_ret_1d,brent_ret_5d,brent_ret_20d,brent_ma_5,brent_ma_20,brent_ma_60,brent_vol_5d,high_low_range,crude_stock_level,gas_stock_level,dist_stock_level,crude_stock_wow_change,gas_stock_wow_change,dist_stock_wow_change,crude_stock_vs_5yr_avg,gas_stock_vs_5yr_avg,dist_stock_vs_5yr_avg,prod_weekly,prod_4w_ma,prod_wow_change,crude_exports,crude_imports,import_4w_ma,net_imports,refinery_run_rate,wti_mm_net_long,wti_mm_net_long_ratio,wti_producer_hedge_ratio,wti_mm_position_change_wow,wti_sentiment_position,DXY_DX_Y_NYB,XLE_XLE,VDE_VDE,IXC_IXC,__DXY____DX_Y_NYB___ret_1d,__XLE____XLE___ret_1d,__VDE____VDE___ret_1d,__IXC____IXC___ret_1d,us10y_yield,us2y_yield,term_spread,bdi_price,bdi_open,bdi_high,bdi_low,bdi_change__,wti_mm_net_long_is_update,wti_mm_net_long_shock_raw,wti_mm_net_long_days_since_update,wti_mm_net_long_avail,wti_mm_net_long_shock_decay,wti_mm_net_long_ratio_is_update,wti_mm_net_long_ratio_shock_raw,wti_mm_net_long_ratio_days_since_update,wti_mm_net_long_ratio_avail,wti_mm_net_long_ratio_shock_decay,wti_producer_hedge_ratio_is_update,wti_producer_hedge_ratio_shock_raw,wti_producer_hedge_ratio_days_since_update,wti_producer_hedge_ratio_avail,wti_producer_hedge_ratio_shock_decay,wti_mm_position_change_wow_is_update,wti_mm_position_change_wow_shock_raw,wti_mm_position_change_wow_days_since_update,wti_mm_position_change_wow_avail,wti_mm_position_change_wow_shock_decay,wti_sentiment_position_is_update,wti_sentiment_position_shock_raw,wti_sentiment_position_days_since_update,wti_sentiment_position_avail,wti_sentiment_position_shock_decay,event_large_move,flag_crude_stock_wow_change,flag_prod_wow_change,flag_wti_mm_position_change_wow,wti_producer_hedge_ratio_lag3,wti_producer_hedge_ratio_lag5,spread_regime,spread_regime_code,spread_extreme,spread_chg_5d,crude_vs5_regime,crude_vs5_regime_code,crude_vs5_extreme,crude_vs5_zscore,news_total_count,news_emb_0,news_emb_1,news_emb_2,news_emb_3,news_emb_4,news_emb_5,news_emb_6,news_emb_7,news_emb_8,news_emb_9,news_emb_10,news_emb_11,news_emb_12,news_emb_13,news_emb_14,news_emb_15,news_emb_16,news_emb_17,news_emb_18,news_emb_19,news_emb_20,news_emb_21,news_emb_22,news_emb_23,news_emb_24,news_emb_25,news_emb_26,news_emb_27,news_emb_28,news_emb_29,news_emb_30,news_emb_31,news_emb_32,news_emb_33,news_emb_34,news_emb_35,news_emb_36,news_emb_37,news_emb_38,news_emb_39,news_emb_40,news_emb_41,news_emb_42,news_emb_43,news_emb_44,news_emb_45,news_emb_46,news_emb_47,news_emb_48,news_emb_49,news_emb_50,news_emb_51,news_emb_52,news_emb_53,news_emb_54,news_emb_55,news_emb_56,news_emb_57,news_emb_58,news_emb_59,news_emb_60,news_emb_61,news_emb_62,news_emb_63,news_uuids,clust_id,clust_count,clust_mean,clust_sum,y_ret,ret_5d,ret_10d,ret_20d
0,2014-01-02,107.779999,95.440002,12.339996,-0.027256,-0.036819,-0.032930,110.790001,110.6705,109.391334,0.011831,0.033680,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.000000,0.000000,0.0,0.00,0.000000,0.0,0.0,0.00,0.0,0.0,0.0,0.00000,0.000000,0.0,0.0,80.629997,55.728363,85.919952,27.177942,0.000000,0.000000,0.000000,0.000000,3.00,0.39,2.61,2113.0,2113.0,2113.0,2113.0,-0.0720,0,0.0,0.0,0,0.0,0,0.0,0.0,0,0.0,0,0.0,0.0,0,0.0,0,0.0,0.0,0,0.0,0,0.0,0.0,0,0.0,0,0,0,0,0.0,0.0,2,1,1,0.0,NaN,0.0,1,0.000000,1.0,-16.985781,1.941138,0.409525,0.716653,1.404438,1.505001,-1.049370,-0.750015,-0.717748,-0.684713,-0.019861,-0.397699,0.108216,1.037515,-0.018697,-0.290782,-0.011142,0.180858,-0.058451,-0.008488,0.300006,0.490811,0.358170,-0.417532,0.461595,-0.015604,-0.014203,0.298583,-0.221794,-0.059309,0.902873,0.280538,0.338775,0.672535,-0.232616,-0.242950,0.079401,-0.132877,0.417334,0.197269,0.065511,0.002837,-0.044323,0.178835,0.334882,-0.299075,0.006763,0.009454,0.458828,-0.403282,0.461847,0.208957,0.177038,-0.108208,-0.052849,-0.502219,-0.685612,-0.246974,-0.458016,-0.169944,0.210896,-0.423135,0.256165,-0.326144,['0d8e95ab-a21d-4755-8ce1-c7ae0190b058'],9,1.0,-0.8,-0.8,-0.008258,-0.012981,-0.006423,-0.016276
1,2014-01-03,106.889999,93.95999